# Milestone 6 — 5-case scored VLM batch + 1 exploratory case (KiPA22)

> **INVALID RESULTS.** The saved outputs of this notebook came from the old blank (saturated-white) input images, caused by a CT intensity-preprocessing bug. They are preserved unchanged for transparency and must not be cited. The export scripts have since been fixed; see `docs/PHASE0_AUDIT.md`.

Model: `google/medgemma-1.5-4b-it` (same checkpoint as Milestone 5), loaded **once**, reused across all cases.

Scored cases: 0, 3, 5, 6, 8 (all `qc_flag = pass_basic`).
Exploratory (not scored): case 2 (`qc_flag = ellipse_outside_bbox`) — run through the same pipeline for a qualitative look, excluded from every averaged statistic.

No fine-tuning, no training, no other cases, deterministic generation (`do_sample=False`, `num_beams=1`, fixed `max_new_tokens`).

**Run cells top to bottom, in order.** Note Step 2's restart requirement below — don't skip it.

## Step 1 — Enable GPU
**Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save.**

In [ ]:
!nvidia-smi

## Step 2 — Install dependencies

In [ ]:
# Pillow is pinned to a known-working version rather than blindly upgraded with
# -U: an uncontrolled Pillow upgrade is a common source of silent, hard-to-debug
# image-decoding differences between runs. transformers/accelerate/bitsandbytes
# are left on -U (matching Milestone 5) since MedGemma needs a recent
# transformers release, but their resolved versions are printed below so the
# run is reproducible from the printed log even without a hard pin.
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub "pillow==11.3.0"

import transformers, accelerate, bitsandbytes, PIL
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("pillow:", PIL.__version__)

### IMPORTANT — restart the runtime now
Installing/upgrading `transformers`/`accelerate` inside a live Colab session can leave stale versions already imported in memory, which later shows up as confusing errors (e.g. `apply_chat_template` signature mismatches, or missing attributes). **After the install cell above finishes, go to Runtime -> Restart session, then continue running from the next cell (Step 3) onward.** You do *not* need to re-run the install cell after restarting — the packages are already installed on disk; the restart just gives you a clean Python process that imports the new versions correctly.

## Step 3 — Log in to Hugging Face (Colab secret, no hardcoded token)
Same as Milestone 5: add your token as a Colab secret named `HF_TOKEN` (key icon, left sidebar), with notebook access enabled.

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    print("Loaded token from Colab secret 'HF_TOKEN'.")
except Exception:
    from getpass import getpass
    hf_token = getpass("Enter your Hugging Face token: ")

login(token=hf_token)
hf_token = None

## Step 4 — Upload and unzip the 6 case images + metadata
Upload `outputs/vlm_batch5/images.zip` from your local repo (generated by `src/prepare_vlm_batch.py`). It contains 6 PNGs (`case_{id}_slice.png`) and `batch_metadata.csv`.

In [ ]:
import zipfile
from google.colab import files

uploaded = files.upload()  # select images.zip
zip_filename = next(iter(uploaded.keys()))

with zipfile.ZipFile(zip_filename) as zf:
    zf.extractall("batch_input")

!ls batch_input

In [ ]:
import csv

cases = []
with open("batch_input/batch_metadata.csv", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        cases.append({
            "case_id": row["case_id"],
            "scored": row["scored"] == "True",
            "pixel_spacing_mm": float(row["pixel_spacing_mm"]),
            "major_mm_ref": float(row["major_mm_ref"]),
            "minor_mm_ref": float(row["minor_mm_ref"]),
            "qc_flag": row["qc_flag"],
            "image_filename": row["image_filename"],
        })

for c in cases:
    print(c)

## Step 5 — Load the model ONCE (bf16 first, 4-bit fallback if needed)
Same checkpoint, same loading code as Milestone 5. This is the only `from_pretrained` call in the whole notebook — every case and every prompt below reuses these `model`/`processor` objects, so the 8.6GB weights are downloaded and loaded into GPU memory exactly once.

In [ ]:
import gc
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/medgemma-1.5-4b-it"

# Guard against doubled GPU memory usage if this cell (or the 4-bit fallback
# below) is accidentally re-run after a partial/previous load.
for _name in ("model", "processor"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print("Loaded in bf16.")

### 4-bit fallback (only run this if the bf16 cell above raised a CUDA out-of-memory error)

In [ ]:
# Only run this cell if Step 5 (bf16) failed with a CUDA out-of-memory error.
import gc
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "google/medgemma-1.5-4b-it"

# Same guard as the bf16 cell: clear any partially-loaded objects first so
# switching from a failed bf16 attempt to 4-bit doesn't leak GPU memory.
for _name in ("model", "processor"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
)
print("Loaded in 4-bit (nf4).")

## Step 6 — Prompt templates (identical to Milestone 5, spacing sentence now per-case)

In [ ]:
def prompt_a():
    return (
        "You are shown an axial CT slice of a kidney. Identify the kidney tumor (if visible) and estimate its size.\n"
        "Report the length of the tumor's longest axis (major axis) and the length of its shortest perpendicular axis (minor axis), in millimeters.\n"
        "Respond only in this exact format:\n"
        '<answer>{"major_mm": X, "minor_mm": Y}</answer>\n'
        "where X and Y are numbers in millimeters."
    )

def prompt_b(pixel_spacing_mm):
    return (
        f"You are shown an axial CT slice of a kidney. This image has a pixel spacing of {pixel_spacing_mm} mm x {pixel_spacing_mm} mm per pixel\n"
        "(each pixel represents that many millimeters of real-world distance, in both the horizontal and vertical directions).\n"
        "Using this scale, identify the kidney tumor (if visible) and estimate its size.\n"
        "Report the length of the tumor's longest axis (major axis) and the length of its shortest perpendicular axis (minor axis), in millimeters.\n"
        "Respond only in this exact format:\n"
        '<answer>{"major_mm": X, "minor_mm": Y}</answer>\n'
        "where X and Y are numbers in millimeters."
    )

print(prompt_a())
print()
print(prompt_b(0.5859))

## Step 7 — Inference helper (deterministic settings)

In [ ]:
MAX_NEW_TOKENS = 300

def run_prompt(image, prompt_text):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device, dtype=model.dtype if hasattr(model, "dtype") else torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
        )
        generation = generation[0][input_len:]

    return processor.decode(generation, skip_special_tokens=True)

## Step 8 — Run all 6 cases x 2 experiments (12 calls, same model instance)

In [ ]:
import os
from PIL import Image

os.makedirs("raw_responses", exist_ok=True)
results = []

for c in cases:
    image = Image.open(os.path.join("batch_input", c["image_filename"])).convert("RGB")

    for exp_name, prompt_text in [
        ("A_no_spacing", prompt_a()),
        ("B_with_spacing", prompt_b(c["pixel_spacing_mm"])),
    ]:
        print(f"Running case {c['case_id']} / {exp_name} ...")
        response = run_prompt(image, prompt_text)

        raw_path = f"raw_responses/case_{c['case_id']}_{exp_name}.txt"
        with open(raw_path, "w") as f:
            f.write(f"=== Prompt ===\n{prompt_text}\n\n=== Raw response ===\n{response}\n")

        results.append({
            "case_id": c["case_id"],
            "qc_flag": c["qc_flag"],
            "scored": c["scored"],
            "experiment": exp_name,
            "pixel_spacing_mm": c["pixel_spacing_mm"],
            "major_mm_ref": c["major_mm_ref"],
            "minor_mm_ref": c["minor_mm_ref"],
            "raw_response": response,
            "raw_response_path": raw_path,
        })

print("\nDone. Ran", len(results), "case/experiment combinations.")

## Step 9 — Parse answers, validate the numbers, compute errors, and per-case spacing_effect

In [ ]:
import re
import json
import math

REFUSAL_MARKERS = ["i cannot", "i can't", "i am not able", "i'm not able", "unable to provide", "cannot provide a diagnosis"]

def parse_answer(raw_text):
    """Extract the <answer>{...}</answer> JSON block.
    Returns (major_mm, minor_mm, status) where status is 'ok', 'refused', or
    'parse_failed'. Numeric validity (finite, > 0) is checked separately below,
    so a syntactically valid but physically nonsensical answer (e.g. 0 or a
    negative number) is not silently accepted here.
    """
    match = re.search(r"<answer>\s*(\{.*?\})\s*</answer>", raw_text, re.DOTALL)
    if not match:
        lowered = raw_text.lower()
        if any(marker in lowered for marker in REFUSAL_MARKERS):
            return None, None, "refused"
        return None, None, "parse_failed"
    try:
        parsed = json.loads(match.group(1))
        major, minor = parsed.get("major_mm"), parsed.get("minor_mm")
        if major is None or minor is None:
            return None, None, "parse_failed"
        return float(major), float(minor), "ok"
    except (json.JSONDecodeError, TypeError, ValueError):
        return None, None, "parse_failed"


def is_valid_numeric(value):
    """A tumor axis length must be finite and strictly positive."""
    return math.isfinite(value) and value > 0


for r in results:
    major_pred, minor_pred, status = parse_answer(r["raw_response"])

    # Numeric validation: JSON parsed fine, but the values themselves must be
    # usable lengths. This only relabels the status -- it never edits the
    # predicted values, so bad numbers stay visible in the CSV instead of
    # being swallowed or corrupting the averages.
    if status == "ok" and not (is_valid_numeric(major_pred) and is_valid_numeric(minor_pred)):
        status = "invalid_numeric"

    r["major_mm_pred"] = major_pred
    r["minor_mm_pred"] = minor_pred
    r["parse_status"] = status

    # Flag (never silently swap) a major axis reported shorter than the minor
    # axis -- a logically inconsistent answer worth surfacing as-is.
    if major_pred is not None and minor_pred is not None:
        r["major_lt_minor_flag"] = major_pred < minor_pred
    else:
        r["major_lt_minor_flag"] = None

    if status == "ok":
        r["major_abs_error"] = abs(major_pred - r["major_mm_ref"])
        r["minor_abs_error"] = abs(minor_pred - r["minor_mm_ref"])
        r["major_rel_error"] = r["major_abs_error"] / r["major_mm_ref"]
        r["minor_rel_error"] = r["minor_abs_error"] / r["minor_mm_ref"]
        r["mae_axes"] = (r["major_abs_error"] + r["minor_abs_error"]) / 2.0
        r["mre_axes"] = (r["major_rel_error"] + r["minor_rel_error"]) / 2.0
    else:
        # Only status == 'ok' rows count toward MAE/MRE -- 'invalid_numeric',
        # 'refused', and 'parse_failed' are all excluded (see Step 10 for the
        # exclusion counts, broken down by reason).
        for k in ["major_abs_error", "minor_abs_error", "major_rel_error", "minor_rel_error", "mae_axes", "mre_axes"]:
            r[k] = None

for r in results:
    print(r["case_id"], r["experiment"], r["parse_status"], "major<minor:", r["major_lt_minor_flag"], r.get("mae_axes"), r.get("mre_axes"))

In [ ]:
# spacing_effect: per case, compare Experiment B's MRE to Experiment A's MRE.
# improved  = MRE_B < MRE_A - SPACING_EFFECT_TOLERANCE
# worsened  = MRE_B > MRE_A + SPACING_EFFECT_TOLERANCE
# unchanged = the two are within the tolerance band (treated as effectively the same)
# undetermined = either experiment failed to parse/validate, so no comparison is possible
SPACING_EFFECT_TOLERANCE = 0.01  # 1 percentage point of relative error

by_case = {}
for r in results:
    by_case.setdefault(r["case_id"], {})[r["experiment"]] = r

spacing_effect_by_case = {}
for case_id, exps in by_case.items():
    a, b = exps.get("A_no_spacing"), exps.get("B_with_spacing")
    if a is None or b is None or a["mre_axes"] is None or b["mre_axes"] is None:
        effect = "undetermined"
    else:
        diff = b["mre_axes"] - a["mre_axes"]
        if diff < -SPACING_EFFECT_TOLERANCE:
            effect = "improved"
        elif diff > SPACING_EFFECT_TOLERANCE:
            effect = "worsened"
        else:
            effect = "unchanged"
    spacing_effect_by_case[case_id] = effect

for r in results:
    r["spacing_effect"] = spacing_effect_by_case[r["case_id"]]

spacing_effect_by_case

## Step 10 — Summary over the 5 SCORED cases only (case 2 reported separately)

In [ ]:
scored_results = [r for r in results if r["scored"]]
exploratory_results = [r for r in results if not r["scored"]]

def avg(vals):
    vals = [v for v in vals if v is not None]
    return sum(vals) / len(vals) if vals else None

def parse_success_rate(rows, experiment):
    subset = [r for r in rows if r["experiment"] == experiment]
    if not subset:
        return None, 0, 0
    n_ok = sum(1 for r in subset if r["parse_status"] == "ok")
    return n_ok / len(subset), n_ok, len(subset)

def exclusion_breakdown(rows, experiment):
    """Why each non-'ok' row in this experiment was excluded from MAE/MRE."""
    subset = [r for r in rows if r["experiment"] == experiment and r["parse_status"] != "ok"]
    counts = {}
    for r in subset:
        counts[r["parse_status"]] = counts.get(r["parse_status"], 0) + 1
    return counts

mae_a = avg([r["mae_axes"] for r in scored_results if r["experiment"] == "A_no_spacing"])
mae_b = avg([r["mae_axes"] for r in scored_results if r["experiment"] == "B_with_spacing"])
mre_a = avg([r["mre_axes"] for r in scored_results if r["experiment"] == "A_no_spacing"])
mre_b = avg([r["mre_axes"] for r in scored_results if r["experiment"] == "B_with_spacing"])

rate_a, ok_a, total_a = parse_success_rate(scored_results, "A_no_spacing")
rate_b, ok_b, total_b = parse_success_rate(scored_results, "B_with_spacing")
excl_a = exclusion_breakdown(scored_results, "A_no_spacing")
excl_b = exclusion_breakdown(scored_results, "B_with_spacing")

scored_case_ids = sorted(set(r["case_id"] for r in scored_results))
n_improved = sum(1 for cid in scored_case_ids if spacing_effect_by_case[cid] == "improved")
n_worsened = sum(1 for cid in scored_case_ids if spacing_effect_by_case[cid] == "worsened")
n_unchanged = sum(1 for cid in scored_case_ids if spacing_effect_by_case[cid] == "unchanged")
n_undetermined = sum(1 for cid in scored_case_ids if spacing_effect_by_case[cid] == "undetermined")

print("=== Scored summary (cases 0, 3, 5, 6, 8) ===")
print(f"Parse success rate, Experiment A: {ok_a}/{total_a}" + (f" ({rate_a:.0%})" if rate_a is not None else ""))
print(f"Parse success rate, Experiment B: {ok_b}/{total_b}" + (f" ({rate_b:.0%})" if rate_b is not None else ""))
print(f"Excluded from scoring, Experiment A: {total_a - ok_a} row(s) -- reasons: {excl_a or 'none'}")
print(f"Excluded from scoring, Experiment B: {total_b - ok_b} row(s) -- reasons: {excl_b or 'none'}")
print()
print(f"Average MAE, no spacing (A):   {mae_a}   (computed over {ok_a} valid prediction(s) of {total_a})")
print(f"Average MAE, with spacing (B): {mae_b}   (computed over {ok_b} valid prediction(s) of {total_b})")
print(f"Average MRE, no spacing (A):   {mre_a}   (computed over {ok_a} valid prediction(s) of {total_a})")
print(f"Average MRE, with spacing (B): {mre_b}   (computed over {ok_b} valid prediction(s) of {total_b})")
print(f"Cases improved by spacing:     {n_improved}")
print(f"Cases worsened by spacing:     {n_worsened}")
print(f"Cases unchanged:               {n_unchanged}")
print(f"Cases undetermined (parse/validation failure): {n_undetermined}")

print("\n=== Exploratory case 2 (NOT included above) ===")
for r in exploratory_results:
    print(r["case_id"], r["experiment"], r["parse_status"], r.get("mae_axes"), r.get("mre_axes"), r["spacing_effect"])

## Step 11 — Save results CSV + raw responses, then download

In [ ]:
import csv as csv_module
from google.colab import files  # imported here too so this cell is self-contained even if run without re-running Step 4

fieldnames = [
    "case_id", "qc_flag", "scored", "experiment", "pixel_spacing_mm",
    "major_mm_pred", "minor_mm_pred", "major_mm_ref", "minor_mm_ref",
    "major_lt_minor_flag",
    "major_abs_error", "minor_abs_error", "major_rel_error", "minor_rel_error",
    "mae_axes", "mre_axes", "parse_status", "spacing_effect", "raw_response_path",
]

with open("vlm_batch5_results.csv", "w", newline="") as f:
    writer = csv_module.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in results:
        writer.writerow({k: r.get(k) for k in fieldnames})

with open("vlm_batch5_summary.csv", "w", newline="") as f:
    writer = csv_module.writer(f)
    writer.writerow(["metric", "value"])
    writer.writerow(["scored_avg_mae_no_spacing", mae_a])
    writer.writerow(["scored_avg_mae_with_spacing", mae_b])
    writer.writerow(["scored_avg_mre_no_spacing", mre_a])
    writer.writerow(["scored_avg_mre_with_spacing", mre_b])
    writer.writerow(["scored_parse_success_rate_A", rate_a])
    writer.writerow(["scored_parse_success_rate_B", rate_b])
    writer.writerow(["scored_excluded_count_A", total_a - ok_a])
    writer.writerow(["scored_excluded_count_B", total_b - ok_b])
    writer.writerow(["scored_cases_improved", n_improved])
    writer.writerow(["scored_cases_worsened", n_worsened])
    writer.writerow(["scored_cases_unchanged", n_unchanged])
    writer.writerow(["scored_cases_undetermined", n_undetermined])

import shutil
shutil.make_archive("raw_responses", "zip", "raw_responses")

print("Saved vlm_batch5_results.csv, vlm_batch5_summary.csv, raw_responses.zip")
files.download("vlm_batch5_results.csv")
files.download("vlm_batch5_summary.csv")
files.download("raw_responses.zip")

## Step 12 — After downloading
Move the three downloaded files into your local repo:
- `results/vlm_batch5_results.csv`
- `results/vlm_batch5_summary.csv`
- unzip `raw_responses.zip` into `outputs/vlm_batch5/raw_responses/`

Milestone 5's files (`outputs/vlm_case0/`, `results/vlm_case0_results.csv`) are untouched by this notebook.